In [ ]:
# Gemma 4 (26B) Supervised Fine-Tuning (SFT) on AWS

**Objective:**
This notebook prepares the verified Mercari dataset and executes Parameter-Efficient Fine-Tuning (PEFT) using Low-Rank Adaptation (LoRA) for the `google/gemma-4-26b` model.

**Instructions:**
1. Run the **Setup & Formatting** cell to load the taxonomy and define the formatting logic.
2. Choose your dataset scale: Run either the **10% Sample** cell (for rapid testing) OR the **100% Full Dataset** cell (for production).
3. Run the **AWS Training Engine** cell to execute the QLoRA training loop.

In [ ]:
# Install AWS-compatible GPU training dependencies
!pip install -q transformers peft trl datasets torch wandb bitsandbytes scikit-learn

import os
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# --- Core Paths & Configuration ---
INPUT_DIR = 'path/to/your/output_directory/Split_Files'
EXCEL_PATH = os.path.join(INPUT_DIR, 'Categorized.xlsx')
TAXONOMY_PATH = os.path.join(INPUT_DIR, 'master_taxonomy.json')

# Output directories for data and weights
DATA_DIR_10 = "./aws_training_data/10_percent"
DATA_DIR_100 = "./aws_training_data/100_percent"
WEIGHTS_OUT_10 = "./aws_weights/gemma4_26b_10_percent"
WEIGHTS_OUT_100 = "./aws_weights/gemma4_26b_100_percent"

for directory in [DATA_DIR_10, DATA_DIR_100, WEIGHTS_OUT_10, WEIGHTS_OUT_100]:
    os.makedirs(directory, exist_ok=True)

# --- Load Assets ---
print("Loading Taxonomy and Excel dataset...")
with open(TAXONOMY_PATH, "r", encoding="utf-8") as f:
    taxonomy = json.load(f)

df = pd.read_excel(EXCEL_PATH)

# --- The Perfect Prompt Builder ---
SYSTEM_INSTRUCTION = (
    "You are a strict classification API.\n"
    "Output ONLY the exact category name from the provided 'Valid Categories' list.\n"
    "NO explanations, NO conversational text, NO punctuation outside of the category name."
)

def format_gemma_chat(system_inst: str, user_prompt: str, assistant_response: str) -> str:
    """Wraps text in the exact control tokens expected by Gemma architecture."""
    return (
        f"<bos><start_of_turn>user\n{system_inst}\n\n{user_prompt}<end_of_turn>\n"
        f"<start_of_turn>model\n{assistant_response}<end_of_turn>\n"
    )

def create_prompts_for_row(row) -> list:
    """Iterates through the 3-tier hierarchy to build up to 3 training examples per item."""
    item_name = str(row.get('item_name', '')).strip()
    c1 = str(row.get('category_1', '')).strip()
    c2 = str(row.get('category_2', '')).strip()
    c3 = str(row.get('category_3', '')).strip()
    
    if len(item_name) > 300 or not c1 or c1.lower() == 'nan':
        return []
        
    training_examples = []
    
    # Level 1
    l1_options = list(taxonomy.keys())
    l1_opts_str = "\n".join([f"- {opt}" for opt in l1_options])
    l1_user_prompt = f"Valid Level 1 Categories:\n{l1_opts_str}\n\nTask: Classify this product into one of the categories above.\nProduct: \"{item_name}\"\nCategory:"
    training_examples.append({"text": format_gemma_chat(SYSTEM_INSTRUCTION, l1_user_prompt, c1)})
    
    # Level 2
    if c1 in taxonomy and c2 and c2.lower() != 'nan':
        l2_data = taxonomy[c1]
        l2_options = list(l2_data.keys()) if isinstance(l2_data, dict) else l2_data
        l2_opts_str = "\n".join([f"- {opt}" for opt in l2_options])
        l2_user_prompt = f"Parent Category Path: {c1}\nValid Level 2 Categories:\n{l2_opts_str}\n\nTask: Classify this product into one of the categories above.\nProduct: \"{item_name}\"\nCategory:"
        training_examples.append({"text": format_gemma_chat(SYSTEM_INSTRUCTION, l2_user_prompt, c2)})
        
        # Level 3
        if isinstance(l2_data, dict) and c2 in l2_data and c3 and c3.lower() != 'nan':
            l3_data = l2_data[c2]
            if isinstance(l3_data, (dict, list)):
                l3_options = list(l3_data.keys()) if isinstance(l3_data, dict) else l3_data
                if len(l3_options) > 0:
                    l3_opts_str = "\n".join([f"- {opt}" for opt in l3_options])
                    l3_user_prompt = f"Parent Category Path: {c1} > {c2}\nValid Level 3 Categories:\n{l3_opts_str}\n\nTask: Classify this product into one of the categories above.\nProduct: \"{item_name}\"\nCategory:"
                    training_examples.append({"text": format_gemma_chat(SYSTEM_INSTRUCTION, l3_user_prompt, c3)})
                    
    return training_examples

def save_jsonl(data: list, filename: str):
    """Streams output cleanly to JSONL."""
    with open(filename, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item) + '\n')

In [ ]:
# --- 10% DATASET PREPARATION ---
print("=== Generating 10% Subset ===")

df_10 = df.sample(frac=0.1, random_state=42)

all_data_10 = []
for _, row in tqdm(df_10.iterrows(), total=len(df_10), desc="Formatting 10% Prompts"):
    all_data_10.extend(create_prompts_for_row(row))

# 80/10/10 Split
train_10, temp_10 = train_test_split(all_data_10, test_size=0.2, random_state=42)
val_10, test_10 = train_test_split(temp_10, test_size=0.5, random_state=42)

print(f"10% Split -> Train: {len(train_10)} | Val: {len(val_10)} | Test: {len(test_10)}")

save_jsonl(train_10, os.path.join(DATA_DIR_10, "train.jsonl"))
save_jsonl(val_10, os.path.join(DATA_DIR_10, "valid.jsonl"))
save_jsonl(test_10, os.path.join(DATA_DIR_10, "test.jsonl"))

print(f"✅ Data saved to: {DATA_DIR_10}")
print(f"🎯 Target Weights Directory will be: {WEIGHTS_OUT_10}")

In [ ]:
# --- 100% DATASET PREPARATION ---
print("=== Generating 100% Full Dataset ===")

all_data_100 = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Formatting 100% Prompts"):
    all_data_100.extend(create_prompts_for_row(row))

# 80/10/10 Split
train_100, temp_100 = train_test_split(all_data_100, test_size=0.2, random_state=42)
val_100, test_100 = train_test_split(temp_100, test_size=0.5, random_state=42)

print(f"100% Split -> Train: {len(train_100)} | Val: {len(val_100)} | Test: {len(test_100)}")

save_jsonl(train_100, os.path.join(DATA_DIR_100, "train.jsonl"))
save_jsonl(val_100, os.path.join(DATA_DIR_100, "valid.jsonl"))
save_jsonl(test_100, os.path.join(DATA_DIR_100, "test.jsonl"))

print(f"✅ Data saved to: {DATA_DIR_100}")
print(f"🎯 Target Weights Directory will be: {WEIGHTS_OUT_100}")